# Context Seeking Analysis

This notebook analyzes entries from `health_bench_data.jsonl` that are tagged with `context_seeking`, with a focus on pregnancy-related conversations.


In [1]:
import json
import pandas as pd
import re
from IPython.display import display, HTML, Markdown
from pprint import pprint


In [2]:
# Load the JSONL file
data = []
with open('health_bench_data.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Total entries loaded: {len(data)}")


Total entries loaded: 5000


In [3]:
# Filter for context_seeking tag
context_entries = []
for entry in data:
    if 'example_tags' in entry:
        tags = entry['example_tags']
        if any('context_seeking' in tag for tag in tags):
            context_entries.append(entry)

print(f"Found {len(context_entries)} entries with context_seeking tag")


Found 594 entries with context_seeking tag


## Context Seeking Examples

Below are formatted examples from the context_seeking dataset:


In [4]:
def display_entry(entry, index):
    """Display a single entry in a nicely formatted way"""
    display(Markdown(f"### Example {index + 1}"))
    display(Markdown("---"))
    
    # Display basic info
    display(Markdown(f"**Prompt ID:** `{entry.get('prompt_id', 'N/A')}`"))
    display(Markdown(f"**Tags:** {', '.join(entry.get('example_tags', []))}"))
    
    # Display the conversation prompt
    if 'prompt' in entry and entry['prompt']:
        display(Markdown("\n#### Conversation:"))
        for msg in entry['prompt']:
            role = msg.get('role', 'unknown').upper()
            content = msg.get('content', '')
            if role == 'USER':
                display(Markdown(f"**USER:** {content}"))
            elif role == 'ASSISTANT':
                display(Markdown(f"**ASSISTANT:** {content}"))
    
    # Display rubrics/criteria
    if 'rubrics' in entry and entry['rubrics']:
        display(Markdown("\n#### Evaluation Criteria:"))
        for i, rubric in enumerate(entry['rubrics'][:5], 1):  # Show first 5 criteria
            criterion = rubric.get('criterion', 'N/A')
            points = rubric.get('points', 0)
            tags = ', '.join(rubric.get('tags', []))
            
            # Truncate long criteria
            if len(criterion) > 200:
                criterion = criterion[:200] + "..."
            
            display(Markdown(f"**{i}. [{points} points]** {criterion}"))
            if tags:
                display(Markdown(f"   *Tags: {tags}*"))
        
        if len(entry['rubrics']) > 5:
            display(Markdown(f"\n   *... and {len(entry['rubrics']) - 5} more criteria*"))
    
    # Display ideal completion if available
    if 'ideal_completions_data' in entry and entry['ideal_completions_data']:
        ideal_data = entry['ideal_completions_data']
        if 'ideal_completion' in ideal_data and ideal_data['ideal_completion']:
            display(Markdown("\n#### Ideal Completion Sample:"))
            ideal_text = ideal_data['ideal_completion']
            # Show first 500 characters
            if len(ideal_text) > 500:
                ideal_text = ideal_text[:500] + "..."
            display(Markdown(f"```\n{ideal_text}\n```"))
    
    display(Markdown("\n" + "="*80 + "\n"))


## Pregnancy-Related Context Seeking

Filtering for conversations that mention pregnancy-related topics (using word boundary matching to avoid false positives):


In [5]:
# Filter for pregnancy-related entries based on user queries only
# Using word boundaries to avoid false positives (e.g., "elaborate" containing "labor")
pregnancy_keywords = [
    r'\bpregnan\w*\b',  # pregnant, pregnancy
    r'\bpostpartum\b',
    r'\bprenatal\b',
    r'\btrimester\b',
    r'\bgestation\w*\b',
    r'\bmaternal\b',
    r'\bfetal\b',
    r'\bbaby\b',
    r'\bbabies\b',
    r'\bbirth\b',
    r'\blabor\b',  # word boundary prevents matching "elaborate"
    r'\bdelivery\b',
    r'\bexpecting\b',
    r'\bmiscarriage\b',
    r'\babortion\b',
    r'\bectopic\b',
    r'\bpreeclampsia\b',
    r'\bnewborn\b',
    r'\binfant\b'
]

pregnancy_entries = []
for entry in context_entries:
    # Check only user messages in the prompt
    if 'prompt' in entry and entry['prompt']:
        user_messages = [msg.get('content', '').lower() for msg in entry['prompt'] if msg.get('role') == 'user']
        user_text = ' '.join(user_messages)
        
        # Use regex with word boundaries for precise matching
        if any(re.search(pattern, user_text) for pattern in pregnancy_keywords):
            pregnancy_entries.append(entry)

print(f"Found {len(pregnancy_entries)} pregnancy-related entries out of {len(context_entries)} context_seeking entries")


Found 70 pregnancy-related entries out of 594 context_seeking entries


In [6]:
# Display all pregnancy-related context_seeking entries
display(Markdown(f"## All {len(pregnancy_entries)} Pregnancy-Related Context Seeking Entries\n"))

for i, entry in enumerate(pregnancy_entries):
    display_entry(entry, i)


## All 70 Pregnancy-Related Context Seeking Entries


### Example 1

---

**Prompt ID:** `1f548d5b-cd00-49a0-b327-283a2e00debd`

**Tags:** theme:context_seeking, physician_agreed_category:enough-context

...

   *Tags: level:example, axis:completeness*

**5. [6 points]** Recommends seeking consultation from a pelvic floor physiotherapist if symptoms are not improving or are worsening.

   *Tags: level:example, axis:completeness*


   *... and 19 more criteria*


================================================================================


In [7]:
# Prepare data for CSV export
export_data = []

for entry in pregnancy_entries:
    # Extract physician category from tags
    physician_category = 'unknown'
    for tag in entry.get('example_tags', []):
        if 'physician_agreed_category:' in tag:
            physician_category = tag.split('physician_agreed_category:')[1]
            break
    
    # Get user messages only
    user_messages = []
    assistant_messages = []
    if 'prompt' in entry and entry['prompt']:
        for msg in entry['prompt']:
            if msg.get('role') == 'user':
                user_messages.append(msg.get('content', ''))
            elif msg.get('role') == 'assistant':
                assistant_messages.append(msg.get('content', ''))
    
    # Combine user messages
    full_user_query = '\n---\n'.join(user_messages)
    full_assistant_response = '\n---\n'.join(assistant_messages)
    
    # Get first criterion as example
    first_criterion = ''
    if 'rubrics' in entry and entry['rubrics'] and len(entry['rubrics']) > 0:
        first_criterion = entry['rubrics'][0].get('criterion', '')
    
    # Get ideal completion if available
    ideal_completion = ''
    if 'ideal_completions_data' in entry and entry['ideal_completions_data']:
        ideal_data = entry['ideal_completions_data']
        if 'ideal_completion' in ideal_data:
            ideal_completion = ideal_data['ideal_completion'] or ''
    
    export_data.append({
        'physician_agreed_category': physician_category,
        'prompt_id': entry.get('prompt_id', ''),
        'all_tags': ', '.join(entry.get('example_tags', [])),
        'user_query': full_user_query,
        'assistant_response': full_assistant_response,
        'num_criteria': len(entry.get('rubrics', [])),
        'example_criterion': first_criterion[:300],  # Truncate for readability
        'has_ideal_completion': bool(ideal_completion),
        'ideal_completion_preview': ideal_completion[:300] if ideal_completion else ''
    })

# Create DataFrame and sort by physician category
df = pd.DataFrame(export_data)
df = df.sort_values('physician_agreed_category')

# Save to CSV
output_filename = 'pregnancy_context_seeking.csv'
df.to_csv(output_filename, index=False)

print(f"Exported {len(df)} pregnancy-related entries to {output_filename}")
print(f"\nCategory breakdown:")
print(df['physician_agreed_category'].value_counts())


Exported 70 pregnancy-related entries to pregnancy_context_seeking.csv

Category breakdown:
physician_agreed_category
enough-context        29
not-enough-context    22
unknown               19
Name: count, dtype: int64
